<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0) 설정
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 200)

DATA_DIR = '/content/drive/MyDrive/hdac_data'            # 팀 공통 경로
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'
OUT_DIR  = f'{DATA_DIR}/output'                           # 집계표(md)만 저장
os.makedirs(OUT_DIR, exist_ok=True)

AGE_MIN  = 19                  # 연구대상: 19세 이상 성인
WEIGHT   = 'wt_tot'            # 건강설문 AND 검진 AND 영양 결합 가중치 (이용지침서 표 6-2 p.28, 변수 설명 p.94)
KEEP_NA8 = True                # BD1_11, BS3_1 의 8(비해당)을 별도 범주로 유지 (False면 결측 처리)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 0-1) 추가 변수 후보 찾기 (원자료에서 이름·설명만 확인; 자료 값은 읽지 않음)
# 1인 가구 / 근로 여부 / 신체활동 변수의 실제 이름은 이용지침서·코드북으로 확정해야 합니다.
import pyreadstat
_, _m = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
KEYWORDS = ['가구원', '경제활동', '근로', '직업', '신체활동', '유산소', '운동']
for kw in KEYWORDS:
    hits = {n: l for n, l in _m.column_names_to_labels.items() if l and kw in l}
    print(f'\n[{kw}] 후보 {len(hits)}개')
    for n, l in list(hits.items())[:12]:
        print(f'  {n}: {l}')


[가구원] 후보 1개
  cfam: 가구원수

[경제활동] 후보 2개
  occp: 직업재분류 및 실업/비경제활동인구상태 코드
  EC1_1: 경제활동 상태

[근로] 후보 4개
  EC_stt_2: 종사상지위: 임금근로자 상세
  EC_wht_23: 주당 평균 근로시간
  EC_wht_5: 주/변형근로시간
  EC_wht_6: 변형근로시간:기타상세

[직업] 후보 3개
  occp: 직업재분류 및 실업/비경제활동인구상태 코드
  EC_occp: 표준직업분류 대분류 코드
  EC_lgw_2: 최장일자리: 직업분류코드+실업/비경활상태

[신체활동] 후보 22개
  BE3_71: 고강도 신체활동 여부: 일
  BE3_72: 고강도 신체활동 일수: 일
  BE3_73: 고강도 신체활동 시간(시간): 일
  BE3_74: 고강도 신체활동 시간(분): 일
  BE3_81: 중강도 신체활동 여부: 일
  BE3_82: 중강도 신체활동 일수: 일
  BE3_83: 중강도 신체활동 시간(시간): 일
  BE3_84: 중강도 신체활동 시간(분): 일
  BE3_91: 신체활동 여부: 장소이동
  BE3_92: 신체활동 일수: 장소이동
  BE3_93: 신체활동 시간(시간): 장소이동
  BE3_94: 신체활동 시간(분): 장소이동

[유산소] 후보 1개
  pa_aerobic: 유산소 신체활동 실천율

[운동] 후보 2개
  LQ_1EQL: EuroQoL: 운동능력
  BE5_1: 1주일간 근력운동 일수


In [3]:
# 0-2) 11주차 추가 변수 이름 (위 후보 목록과 코드북으로 확인한 뒤 필요하면 고치기)
EXTRA_VARS = {
    'hh':   'cfam',        # 가구원수 -> 1인 가구 여부 만들기
    'work': 'EC1_1',       # 현재 경제활동(근로) 여부
    'pa':   'pa_aerobic',  # 유산소 신체활동 실천 여부
}

In [4]:
# 1) 변수 정의표의 변수만 불러오기
!pip -q install pyreadstat
import pyreadstat
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT] + [v.lower() for v in EXTRA_VARS.values()]   # 11주차: 추가 변수
_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
absent = [c for c in NEEDED if c not in low2real]
assert not absent, f'원자료에 없는 변수: {absent}'
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()
print(f'불러온 자료: {len(df):,}행 x {df.shape[1]}열 | 조사연도:', sorted(df['year'].dropna().unique()))

불러온 자료: 6,997행 x 16열 | 조사연도: [np.float64(2024.0)]


In [5]:
# ==============================================================================
# [셀 1] 10주차 전처리 및 데이터 로드
# ==============================================================================
from google.colab import drive
import os
import numpy as np
import pandas as pd
import pyreadstat

# 1) 구글 드라이브 연결
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'

AGE_MIN = 19                  # 연구대상: 19세 이상 성인
WEIGHT = 'wt_tot'             # 가중치 (건강설문 AND 검진 AND 영양 결합 가중치)

# 2) 필요한 변수만 불러오기
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT]

_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()

# 3) 노출·결과·공변량 및 분석 대상(analytic) 생성 함수 정의
def num(s): return pd.to_numeric(s, errors='coerce')

# 노출: 외식 빈도 3범주
EXP_VALID = [1, 2, 3, 4, 5, 6, 7]
GRP_MAP   = {5: 1, 6: 1, 7: 1, 4: 2, 1: 3, 2: 3, 3: 3}
df['exp_raw']   = num(df['l_out_fq'])
df['exp_valid'] = df['exp_raw'].isin(EXP_VALID)
df['grp']       = df['exp_raw'].where(df['exp_valid']).map(GRP_MAP)

# 결과, 연령, 가중치
df['bmi']     = num(df['he_bmi'])
df['age_n']   = num(df['age'])
df['w']       = num(df[WEIGHT])

# 공변량 원시 데이터 매핑
df['c_sex']   = num(df['sex']).map({1: '남성', 2: '여성'})
df['c_educ']  = num(df['educ']).mask(num(df['educ']).isin([88, 99])).map({1: '초졸 이하', 2: '초졸 이하', 3: '초졸 이하', 4: '중졸', 5: '고졸', 6: '대졸 이상', 7: '대졸 이상', 8: '대졸 이상'})
df['c_incm']  = num(df['incm']).mask(num(df['incm']).isin([8, 9, 99])).map({1: '하', 2: '중하', 3: '중상', 4: '상'})
df['c_bd1_11']= num(df['bd1_11']).mask(num(df['bd1_11']).isin([9, 99])).map({1: '전혀 마시지 않음', 2: '월 1회 미만', 3: '월 1회 정도', 4: '월 2-4회', 5: '주 2-3회', 6: '주 4회 이상', 8: '비해당(코드 8)'})
df['c_bs3_1'] = num(df['bs3_1']).mask(num(df['bs3_1']).isin([9, 99])).map({1: '매일 흡연', 2: '가끔 흡연', 3: '과거 흡연(현재 비흡연)', 8: '비해당(코드 8)'})

# 설계변수 및 analytic(분석 대상) 마스크 생성
psu_str = df['psu'].astype(str).str.strip()
df['psu_ok']    = df['psu'].notna() & ~psu_str.isin(['', 'nan', 'None'])
df['design_ok'] = df['w'].notna() & (df['w'] > 0) & df['kstrata'].notna() & df['psu_ok']

df['s1'] = True
df['s2'] = df['age_n'] >= AGE_MIN
df['s3'] = df['s2'] & df['exp_valid']
df['s4'] = df['s3'] & df['bmi'].notna()
df['s5'] = df['s4'] & df['design_ok']
df['analytic'] = df['s5']

print(f"전처리 완료! 최종 분석 대상(analytic=True): {int(df['analytic'].sum()):,}명")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전처리 완료! 최종 분석 대상(analytic=True): 5,765명


#C-1. 화살표 및 인과 근거 표 출력 코드

In [6]:
# ==============================================================================
# C-1. 화살표 및 인과 근거 표 출력 코드
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

# 1. 데이터 정의
data = [
    {
        "화살표": "연령 → 교육, 소득, 근로, 1인 가구",
        "근거": "출생 코호트에 따라 학력이 다름(시간 순서: 연령·코호트가 먼저). 생애주기에 따라 근로·소득·거주 형태가 달라짐"
    },
    {
        "화살표": "연령, 성별 → 외식 빈도",
        "근거": "Table 1에서 저빈도군은 평균 연령이 높고(57세 대 43세), 고빈도군은 남성 비율이 높음(66%)"
    },
    {
        "화살표": "연령, 성별 → BMI",
        "근거": "연령·성별에 따른 체성분과 BMI 분포 차이는 선행연구에서 일관됨"
    },
    {
        "화살표": "교육 → 소득, 근로",
        "근거": "학력이 취업과 소득에 선행"
    },
    {
        "화살표": "교육, 소득 → 외식 빈도",
        "근거": "Table 1에서 고빈도군은 대졸 이상(71%)과 소득 상위가 많음 (자료 확인)"
    },
    {
        "화살표": "교육, 소득 → BMI",
        "근거": "사회경제적 수준과 비만의 관련은 선행연구에서 보고됨"
    },
    {
        "화살표": "지역 → 외식 빈도, BMI, 소득, 근로",
        "근거": "도시·농촌에 따른 외식 환경(식당 접근성)과 일자리 구조 차이"
    },
    {
        "화살표": "근로 → 외식 빈도",
        "근거": "직장인은 점심·회식 등으로 외식 기회가 많음"
    },
    {
        "화살표": "근로 → BMI",
        "근거": "근로 형태(좌식, 교대근무)가 체중에 영향"
    },
    {
        "화살표": "1인 가구 → 외식 빈도, BMI",
        "근거": "혼자 살면 조리 대신 외식·간편식을 택하기 쉽고, 식사 패턴이 BMI에 영향"
    },
    {
        "화살표": "U → 외식 빈도, 신체활동, 흡연",
        "근거": "건강 지향 성향은 식생활·운동·흡연에 공통으로 작용한다고 가정 (미측정)"
    },
    {
        "화살표": "신체활동, 흡연 → BMI",
        "근거": "에너지 소비와 체중 조절에 대한 영향"
    },
    {
        "화살표": "외식 → 음주 → BMI",
        "근거": "외식·회식 자리에서 음주가 늘고, 알코올 열량이 BMI에 영향 (매개)"
    },
    {
        "화살표": "외식 → 에너지 섭취 → BMI",
        "근거": "연구 가설의 핵심 경로 (매개)"
    }
]

# 2. DataFrame 생성
df_c1 = pd.DataFrame(data)

# 3. 코랩 화면에 예쁜 HTML 표로 출력
print("=== [C-1] 화살표 및 인과 근거 표 ===")
display(HTML(df_c1.to_html(index=False, escape=False)))

=== [C-1] 화살표 및 인과 근거 표 ===


화살표,근거
"연령 → 교육, 소득, 근로, 1인 가구",출생 코호트에 따라 학력이 다름(시간 순서: 연령·코호트가 먼저). 생애주기에 따라 근로·소득·거주 형태가 달라짐
"연령, 성별 → 외식 빈도","Table 1에서 저빈도군은 평균 연령이 높고(57세 대 43세), 고빈도군은 남성 비율이 높음(66%)"
"연령, 성별 → BMI",연령·성별에 따른 체성분과 BMI 분포 차이는 선행연구에서 일관됨
"교육 → 소득, 근로",학력이 취업과 소득에 선행
"교육, 소득 → 외식 빈도",Table 1에서 고빈도군은 대졸 이상(71%)과 소득 상위가 많음 (자료 확인)
"교육, 소득 → BMI",사회경제적 수준과 비만의 관련은 선행연구에서 보고됨
"지역 → 외식 빈도, BMI, 소득, 근로",도시·농촌에 따른 외식 환경(식당 접근성)과 일자리 구조 차이
근로 → 외식 빈도,직장인은 점심·회식 등으로 외식 기회가 많음
근로 → BMI,"근로 형태(좌식, 교대근무)가 체중에 영향"
"1인 가구 → 외식 빈도, BMI","혼자 살면 조리 대신 외식·간편식을 택하기 쉽고, 식사 패턴이 BMI에 영향"


#C-1. 공변량별 역할 및 보정 여부 표 출력 코드

In [7]:
# ==============================================================================
# C-1. 공변량별 역할 및 보정 여부 표 출력 코드
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

# 1. 데이터 정의 (이미지 내용 반영)
data_role = [
    {
        "변수명 (원변수)": "외식 빈도 <br>( L_OUT_FQ )",
        "DAG 상의 역할": "노출",
        "근거": "• 연구의 독립변수(원인 후보)<br>• 주간 섭취 빈도에 따라 저·중·고빈도로 분류됨",
        "총효과 모형 보정 여부": "- (노출)"
    },
    {
        "변수명 (원변수)": "체질량지수 BMI <br>( HE_BMI )",
        "DAG 상의 역할": "결과",
        "근거": "• 연구의 종속변수(결과)<br>• 외식으로 인한 열량 섭취 증가의 최종 건강 결과 지표",
        "총효과 모형 보정 여부": "- (결과)"
    },
    {
        "변수명 (원변수)": "성별 ( sex )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 생물학적 특성은 외식 빈도와 대사율·체지방 분포(BMI)에 선행하여 영향을 미침",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "거주 지역 ( region )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 지역별 외식 인프라(식당 접근성)와 거주민의 식생활 환경 차이가 외식 빈도와 BMI에 영향을 미침",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "교육수준 ( educ )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 사회경제적 지위(SES)는 식습관 및 건강 인식을 결정하며 외식 빈도와 비만 유병률에 영향을 미침",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "소득 4분위 ( incm )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 경제적 자원(소득 수준)은 외식 소비 여력과 식품 선택에 직접 작용하며 BMI와도 연관됨",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "음주 빈도 ( BD1_11 )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 외식 상황(회식, 모임 등)과 밀접하게 연동되어 외식 빈도에 영향을 미치며, 알코올 열량을 통해 BMI 대사에 관여함",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "현재 흡연 ( BS3_1 )",
        "DAG 상의 역할": "교란",
        "근거": "근거: 개인의 건강 관련 라이프스타일 지표로서 식생활 패턴(외식) 및 체중 조절 기전에 공통으로 작용함",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "1인 가구 ( cfam )",
        "DAG 상의 역할": "교란",
        "근거": "식사 방식과 BMI에 모두 영향",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "근로 여부 ( EC1_1 )",
        "DAG 상의 역할": "교란",
        "근거": "외식 기회와 BMI에 모두 영향",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "신체활동 <br>( pa_aerobic )",
        "DAG 상의 역할": "교란",
        "근거": "U의 영향을 BMI까지 전달하는 경로 위 변수",
        "총효과 모형 보정 여부": "보정함"
    },
    {
        "변수명 (원변수)": "에너지 섭취",
        "DAG 상의 역할": "매개",
        "근거": "연구 가설의 핵심 경로",
        "총효과 모형 보정 여부": "보정함"
    }
]

# 2. DataFrame 생성
df_role = pd.DataFrame(data_role)

# 3. 코랩 화면에 HTML 표로 출력
print("=== [C-1] 공변량별 역할 및 보정 여부 표 ===")
display(HTML(df_role.to_html(index=False, escape=False)))

=== [C-1] 공변량별 역할 및 보정 여부 표 ===


변수명 (원변수),DAG 상의 역할,근거,총효과 모형 보정 여부
외식 빈도 ( L_OUT_FQ ),노출,• 연구의 독립변수(원인 후보)• 주간 섭취 빈도에 따라 저·중·고빈도로 분류됨,- (노출)
체질량지수 BMI ( HE_BMI ),결과,• 연구의 종속변수(결과)• 외식으로 인한 열량 섭취 증가의 최종 건강 결과 지표,- (결과)
성별 ( sex ),교란,근거: 생물학적 특성은 외식 빈도와 대사율·체지방 분포(BMI)에 선행하여 영향을 미침,보정함
거주 지역 ( region ),교란,근거: 지역별 외식 인프라(식당 접근성)와 거주민의 식생활 환경 차이가 외식 빈도와 BMI에 영향을 미침,보정함
교육수준 ( educ ),교란,근거: 사회경제적 지위(SES)는 식습관 및 건강 인식을 결정하며 외식 빈도와 비만 유병률에 영향을 미침,보정함
소득 4분위 ( incm ),교란,근거: 경제적 자원(소득 수준)은 외식 소비 여력과 식품 선택에 직접 작용하며 BMI와도 연관됨,보정함
음주 빈도 ( BD1_11 ),교란,"근거: 외식 상황(회식, 모임 등)과 밀접하게 연동되어 외식 빈도에 영향을 미치며, 알코올 열량을 통해 BMI 대사에 관여함",보정함
현재 흡연 ( BS3_1 ),교란,근거: 개인의 건강 관련 라이프스타일 지표로서 식생활 패턴(외식) 및 체중 조절 기전에 공통으로 작용함,보정함
1인 가구 ( cfam ),교란,식사 방식과 BMI에 모두 영향,보정함
근로 여부 ( EC1_1 ),교란,외식 기회와 BMI에 모두 영향,보정함


#완전 사례 표시

In [8]:
# ==============================================================================
# [셀 2] 완전 사례(Complete Case) 분석 대상 인원 확인 및 Table 1 비교
# ==============================================================================
import pandas as pd

# 1. Table 1의 분석 대상 인원 (analytic=True)
n_table1 = int(df['analytic'].sum())

# 2. 보정변수(공변량) 원본 변수명 리스트 설정
# (다변량 회귀분석 모형 2에 투입되는 필수 보정변수들)
adjustment_cols = ['sex', 'region', 'educ', 'incm', 'bd1_11', 'bs3_1', 'age_n']

# 3. 완전 사례(모든 보정변수에 결측치가 없는 사람) 마스크 생성
df['complete_case'] = df['analytic'] & df[adjustment_cols].notna().all(axis=1)

n_complete = int(df['complete_case'].sum())
diff_count = n_table1 - n_complete

# 4. 결과 출력
print("=== [완전 사례 분석 대상 인원 확인] ===")
print(f"1) Table 1 분석 대상 인원 (analytic=True): {n_table1:,} 명")
print(f"2) 보정변수 완전 사례 인원 (Complete Case): {n_complete:,} 명")
print(f"3) 인원 차이: {diff_count:,} 명 (Table 1 대비 공변량 결측으로 제외된 수)")
print("========================================")

=== [완전 사례 분석 대상 인원 확인] ===
1) Table 1 분석 대상 인원 (analytic=True): 5,765 명
2) 보정변수 완전 사례 인원 (Complete Case): 5,737 명
3) 인원 차이: 28 명 (Table 1 대비 공변량 결측으로 제외된 수)


#C-2 복합표본 가중 선형회귀분석 (모형 1 & 모형 2) 및 Table 2 출력 코드

In [9]:
# ==============================================================================
# [C-2 최종 수정] 모형 1 및 모형 2 결과 매트릭스 생성 코드
# ==============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 분석 공변량 정의 (모형 2용 최소 보정집합)
adjustment_cols = ['sex', 'region', 'educ', 'incm', 'bd1_11', 'bs3_1', 'age_n']

# 2. Table 1 분석 대상 인원 (analytic=True)
n_table1 = int(df['analytic'].sum())

# 3. 완전 사례(Complete Case) 마스크 생성
valid_mask = (
    df['analytic'] &
    df['grp'].notna() &
    df['bmi'].notna() &
    df[adjustment_cols].notna().all(axis=1) &
    df['design_ok']
)
sub_df = df[valid_mask].copy()
n_complete = len(sub_df)
diff_count = n_table1 - n_complete

# 범주형 변수 처리 (Ref: 저빈도 외식군 = 3)
sub_df['grp'] = pd.Categorical(sub_df['grp'], categories=[3, 2, 1], ordered=True)

# 4. 회귀분석 함수
def fit_wls(formula, data):
    model = smf.wls(formula, data=data, weights=data['w'])
    return model.fit(cov_type='cluster', cov_kwds={'groups': data['psu']})

# 모형 1 및 모형 2 적합
res_m1 = fit_wls('bmi ~ C(grp, Treatment(reference=3))', sub_df)
res_m2 = fit_wls('bmi ~ C(grp, Treatment(reference=3)) + C(sex) + C(region) + C(educ) + C(incm) + C(bd1_11) + C(bs3_1) + age_n', sub_df)

# 5. 결과 포맷팅 함수
def format_result(res, cat_name):
    if cat_name == 'ref':
        return "Ref. (-)"

    param_idx = None
    for idx, name in enumerate(res.params.index):
        if cat_name in name:
            param_idx = idx
            break

    if param_idx is None:
        return "-"

    coef = res.params[param_idx]
    se = res.bse[param_idx]
    ci_low = res.conf_int().iloc[param_idx, 0]
    ci_high = res.conf_int().iloc[param_idx, 1]
    pval = res.pvalues[param_idx]
    pval_str = f"{pval:.3f}" if pval >= 0.001 else "< 0.001"

    return f"β = {coef:.3f} (SE = {se:.3f})<br>95% CI: [{ci_low:.3f}, {ci_high:.3f}]<br>p-value = {pval_str}"

# 6. Table 2 최종 데이터프레임 구성 (행=모형, 열=노출 범주)
table2_rows = [
    {
        "모형": "모형 1 (단변량)",
        "분석 인원 (N)": f"{n_complete:,}명<br>(Table 1 대비 {diff_count}명 차이)",
        "저빈도 외식군 (주 1-2회 이하)": format_result(res_m1, 'ref'),
        "중빈도 외식군 (주 3-4회)": format_result(res_m1, '[T.2]'),
        "고빈도 외식군 (주 5회 이상)": format_result(res_m1, '[T.1]')
    },
    {
        "모형": "모형 2 (다변량 보정)",
        "분석 인원 (N)": f"{n_complete:,}명<br>(Table 1 대비 {diff_count}명 차이)",
        "저빈도 외식군 (주 1-2회 이하)": format_result(res_m2, 'ref'),
        "중빈도 외식군 (주 3-4회)": format_result(res_m2, '[T.2]'),
        "고빈도 외식군 (주 5회 이상)": format_result(res_m2, '[T.1]')
    }
]

df_table2_matrix = pd.DataFrame(table2_rows)

# 7. 화면 출력
print("=== [Table 2] 외식 빈도에 따른 체질량지수(BMI) 복합표본 가중 선형회귀분석 결과 ===")
display(df_table2_matrix)

=== [Table 2] 외식 빈도에 따른 체질량지수(BMI) 복합표본 가중 선형회귀분석 결과 ===


/tmp/ipykernel_20419/1013919118.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coef = res.params[param_idx]
/tmp/ipykernel_20419/1013919118.py:54: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se = res.bse[param_idx]
/tmp/ipykernel_20419/1013919118.py:57: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pval = res.pvalues[param_idx]
/tmp/ipykernel_20419/1013919118.py:53: FutureWarning: Series.__getitem__ treating keys as position

,모형,분석 인원 (N),저빈도 외식군 (주 1-2회 이하),중빈도 외식군 (주 3-4회),고빈도 외식군 (주 5회 이상)
0,모형 1 (단변량),"5,737명<br>(Table 1 대비 28명 차이)",Ref. (-),"β = -0.570 (SE = 0.212)<br>95% CI: [-0.985, -0...","β = -0.693 (SE = 0.137)<br>95% CI: [-0.961, -0..."
1,모형 2 (다변량 보정),"5,737명<br>(Table 1 대비 28명 차이)",Ref. (-),"β = -0.300 (SE = 0.207)<br>95% CI: [-0.705, 0....","β = -0.489 (SE = 0.149)<br>95% CI: [-0.782, -0..."


#각주

1. 설계 및 분석 모형: 국민건강영양조사의 복합표본설계 요소(층화변수 kstrata, 집락변수 psu, 가중치 wt_tot)를 반영한 가중 선형회귀모형(WLS, 선형 회귀)을 적합하였으며, 표준오차(SE)는 집락 단위의 강건 표준오차를 적용함.

2. 도메인 처리 및 결측 처리: 분석 대상 밖의 데이터는 도메인 처리 원칙에 따라 완전 사례 서브셋을 구축하여 분석하였으며, 모형 1과 모형 2 모두 동일한 데이터셋을 사용함.

3. 결측 처리 방법: 노출(외식 빈도), 결과(BMI) 및 필수 보정변수(성별, 거주 지역, 교육수준, 소득 4분위, 음주 빈도, 현재 흡연 여부, 연령)에 결측치가 없는 완전 사례만을 최종 분석에 활용함.

4. 모형 종류 및 보정변수 목록:
   - 모형 1 (Unadjusted): 외식 빈도 범주 단독 투입 (미보정 단변량 모형)
   - 모형 2 (Adjusted): C-1 DAG 최소 보정집합(성별, 거주 지역, 교육수준, 소득 4분위, 음주 빈도, 현재 흡연 여부, 연령) 보정한 다변량 모형

5. 분석 인원: Table 1 분석 대상 인원은 총 5,765명이며, 공변량 결측(28명)이 제외된 모형 1 및 모형 2의 최종 분석 인원은 총 5,737명임.

6. 자료 이름과 연도: 질병관리청 국민건강영양조사 제9기 2024년